# CI/CD — Git Folders, Automation Bundles, Databricks CLI

## What's covered

- **Databricks Git Folders** (formerly **Databricks Repos**) — branching, committing, pushing, opening PRs from the workspace UI
- **Automation Bundles** — also called **Declarative Automation Bundles**, formerly **Databricks Asset Bundles (DABs)** — the production way to package and promote everything (jobs, pipelines, notebooks, schemas)
- The `databricks.yml` file — **resources**, **targets**, **variables**, **environment overrides**
- The **Databricks CLI** — `databricks bundle validate / deploy / run / destroy`
- **Workflow: dev → test → prod** — one codebase, three targets
- **CI integration** — running `bundle validate` and `bundle deploy` from GitHub Actions / Azure DevOps
- **Run-as identity** — bundles deploy and execute as a service principal in non-dev environments
- The decision sheet for what belongs in a bundle vs. a notebook vs. an external repo

## Reference domain

The bank's `fintech-platform` Git repo holds:

- Notebooks for ingestion, silver build, gold modelling
- The Lakeflow Spark Declarative Pipeline (from notebook 05)
- The `fintech_nightly_ingest` Lakeflow Job (from notebook 06)
- A single `databricks.yml` that promotes the whole bundle across **dev**, **test**, and **prod** Databricks workspaces

## The problem CI/CD solves on Databricks

Before bundles, promoting a pipeline from dev to prod looked like one of two bad options.

**Option A — point and click.** Engineers cloned notebooks across workspaces by hand, copy-pasted job JSON, updated paths manually. Drift was inevitable: prod ran a job spec slightly different from dev, and the two diverged over months.

**Option B — REST API scripts.** Teams wrote bash that called the Jobs API and the Workspace API to upload notebooks, create jobs, configure pipelines. Workable, but every team reinvented the same plumbing differently. There was no canonical way to describe "this pipeline plus this job plus these notebooks, deployed as one unit."

**Automation Bundles** solve this by giving you a single declarative file (`databricks.yml`) that describes every resource in your project — and a CLI that validates, deploys, and runs that bundle against any target workspace. The same source tree promotes from dev → test → prod by changing one CLI flag.

## Databricks Git Folders — what the exam wants you to know

**Databricks Git Folders** (formerly **Databricks Repos**) is a workspace folder backed by a Git remote — GitHub, GitLab, Azure DevOps, Bitbucket.

What you do from the workspace UI:

- **Clone** a Git repo into a workspace folder.
- **Switch branches** — checkout an existing branch or create a new one.
- **Commit** local changes with a message.
- **Push** to the remote.
- **Pull** updates from the remote.
- **Open a pull request** — the UI links out to the remote (GitHub etc.) to open the PR.

**Two facts the exam tests:**

- Git Folders are the **right answer** to "how do engineers commit notebook changes from inside Databricks." The legacy alternative — exporting notebooks and manually adding them to a Git repo — is wrong on a modern exam.
- Workspace files outside Git Folders are *not* under source control. Production code belongs in a Git Folder (or in a bundle that gets deployed to a workspace path).

**Naming watch:** *Databricks Git Folders* (current) vs. *Databricks Repos* (legacy). Same feature.

# Automation Bundles — the deployment unit

**Automation Bundles** — also called **Declarative Automation Bundles** — package every Databricks resource in your project as code. A bundle has:

- A root **`databricks.yml`** — the manifest.
- **Resource definitions** — jobs, Lakeflow Spark Declarative Pipelines, ML experiments, models, dashboards, volumes, schemas — declared in YAML.
- **Source files** — notebooks, Python files, SQL files, dbt projects — referenced from the manifest.
- **Targets** — named deployments (`dev`, `test`, `prod`) with environment-specific values.
- **Variables** — values you parametrise per target (catalog name, cluster size, schedule).

**Rename watch.** *Databricks Asset Bundles (DABs)* is the original name. *Automation Bundles* and *Declarative Automation Bundles* are the current names. The exam uses any of them.

```text
  fintech-platform/
  ├── databricks.yml                ← the manifest
  ├── resources/
  │   ├── job_nightly.yml           ← Lakeflow Job definitions
  │   └── pipeline_card_etl.yml     ← Lakeflow Spark Declarative Pipeline
  ├── notebooks/
  │   ├── ingest_cards.ipynb
  │   ├── silver_build.ipynb
  │   └── notify.ipynb
  ├── src/
  │   └── pipeline.py               ← LSDP Python module
  └── .github/workflows/deploy.yml  ← CI pipeline that runs bundle deploy
```

## The `databricks.yml` manifest

The manifest's three required top-level sections are `bundle`, `resources`, and `targets`.

```yaml
 bundle:
   name: fintech-platform

 variables:
   catalog:
     description: "Target UC catalog"
     default: fintech_dev
   node_type_id:
     description: "Worker node SKU"
     default: i3.xlarge

 include:
   - resources/*.yml

 targets:
   dev:
     mode: development
     workspace:
       host: https://dev.cloud.databricks.com
     variables:
       catalog: fintech_dev
       node_type_id: i3.xlarge

   test:
     mode: development
     workspace:
       host: https://test.cloud.databricks.com
     variables:
       catalog: fintech_test
       node_type_id: i3.xlarge

   prod:
     mode: production
     workspace:
       host: https://prod.cloud.databricks.com
     run_as:
       service_principal_name: fintech-platform-sp
     variables:
       catalog: fintech_prod
       node_type_id: i3.2xlarge
```

**Three concepts the exam wants you to recognise:**

- **Variables** — single declaration, target-specific values. Pass them into resources with `${var.catalog}`.
- **Targets** — named deployments. Each has its own workspace, identity, and variable overrides.
- **Mode** — `development` (cluster stays alive between runs, names get a username prefix) vs. `production` (cluster terminates, names are exact). Prod targets should always be `mode: production` and run as a **service principal**, not a developer.

## Resources — jobs, pipelines, and everything else

Each resource type has a schema that mirrors the corresponding Databricks REST API. A job uses the same fields you'd send to `POST /api/2.1/jobs/create` (notebook 06).

**`resources/job_nightly.yml`:**

```yaml
 resources:
   jobs:
     fintech_nightly_ingest:
       name: fintech_nightly_ingest
       schedule:
         quartz_cron_expression: "0 0 2 * * ?"
         timezone_id: "UTC"
       email_notifications:
         on_failure: ["data-platform-oncall@bank.example"]
       parameters:
         - name: processing_date
           default: "{{job.start_time.iso_date}}"
       tasks:
         - task_key: ingest_cards
           notebook_task:
             notebook_path: ../notebooks/ingest_cards.ipynb
             base_parameters:
               catalog: ${var.catalog}
           job_cluster_key: shared_etl_cluster
           max_retries: 3
         - task_key: silver_build
           depends_on: [{task_key: ingest_cards}]
           notebook_task:
             notebook_path: ../notebooks/silver_build.ipynb
             base_parameters:
               catalog: ${var.catalog}
           job_cluster_key: shared_etl_cluster
       job_clusters:
         - job_cluster_key: shared_etl_cluster
           new_cluster:
             spark_version: "14.3.x-scala2.12"
             node_type_id: ${var.node_type_id}
             num_workers: 4
```

**`resources/pipeline_card_etl.yml`** — a Lakeflow Spark Declarative Pipeline:

```yaml
 resources:
   pipelines:
     card_etl:
       name: card_etl_${bundle.target}
       target: ${var.catalog}.gold
       libraries:
         - notebook: { path: ../src/pipeline.py }
       continuous: false
       development: true
       channel: CURRENT
```

Notice `${var.catalog}` and `${bundle.target}` — values flow from the manifest into every resource.

## Variables and environment overrides — one codebase, three behaviours

The same `databricks.yml` and the same resource files deploy three different sets of behaviour. The mechanism is **variable resolution per target**.

| Variable | dev | test | prod |
|---|---|---|---|
| `catalog` | `fintech_dev` | `fintech_test` | `fintech_prod` |
| `node_type_id` | `i3.xlarge` (small) | `i3.xlarge` | `i3.2xlarge` (bigger) |
| `mode` | `development` | `development` | `production` |
| `run_as` | the developer | the developer | service principal |
| `schedule` (per env override) | every 4 hrs (lower SLA) | every 4 hrs | nightly 02:00 UTC |

**Override patterns:** at the target level you can override any resource field, not just variables. The prod target can replace the schedule, the cluster size, the email recipients — anything.

**The exam pattern:** *"a team needs to promote the same codebase to dev, test, and prod with different catalog names and cluster sizes."* The answer is **bundle variables + targets**.

## The Databricks CLI — `bundle validate / deploy / run / destroy`

Four CLI subcommands cover the entire bundle workflow.

**`databricks bundle validate -t <target>`** — parses `databricks.yml`, resolves variables for the chosen target, checks all resource references, and emits the *fully-resolved JSON* that would be sent to the API. Run it first; run it in CI on every PR.

**`databricks bundle deploy -t <target>`** — uploads notebooks and source files to the workspace, creates/updates every resource (jobs, pipelines, volumes, schemas) so the workspace matches the manifest. Idempotent — re-deploying the same bundle is a no-op when nothing changed.

**`databricks bundle run <resource_key> -t <target>`** — kicks off a one-off run of a specific resource (a job or a pipeline) and tails its output. The CI equivalent of clicking *Run now* in the UI.

**`databricks bundle destroy -t <target>`** — removes all resources the bundle owns from the target workspace. Use carefully; use it in CI to tear down ephemeral test environments.

**Typical local workflow:**

```bash
 databricks bundle validate -t dev
 databricks bundle deploy   -t dev
 databricks bundle run      fintech_nightly_ingest -t dev
```

**Auth.** The CLI authenticates via:

- An **OAuth user login** (`databricks auth login`) for local dev.
- An **OAuth machine-to-machine** (service principal) token in CI — set `DATABRICKS_HOST` and `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET`.
- A **PAT** (personal access token) — older pattern, still works, less preferred.

## CI workflow — promotion dev → test → prod

The end-to-end shape with GitHub Actions:

```text
                              ┌── PR opened ───────────────────┐
                              │                                │
 developer commits ──► push ──┤                                ▼
                              │              ┌────────────────────────────────────┐
                              │              │  GH Actions: validate + lint       │
                              │              │   • databricks bundle validate -t dev │
                              │              │   • databricks bundle validate -t test │
                              │              │   • databricks bundle validate -t prod │
                              │              └────────────────────────────────────┘
                              │                                │
                              ▼                                ▼
                       PR review + merge ──► main branch ──► GH Actions: deploy
                                                              • databricks bundle deploy -t test
                                                              • run smoke job, wait for green
                                                              • databricks bundle deploy -t prod
```

**A minimal `.github/workflows/deploy.yml`:**

```yaml
 name: Deploy bundle
 on:
   push: { branches: [main] }
 jobs:
   deploy:
     runs-on: ubuntu-latest
     env:
       DATABRICKS_HOST:          ${{ secrets.DATABRICKS_HOST }}
       DATABRICKS_CLIENT_ID:     ${{ secrets.DATABRICKS_CLIENT_ID }}
       DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
     steps:
       - uses: actions/checkout@v4
       - uses: databricks/setup-cli@main
       - run: databricks bundle validate -t test
       - run: databricks bundle deploy   -t test
       - run: databricks bundle run smoke_test_job -t test
       - run: databricks bundle validate -t prod
       - run: databricks bundle deploy   -t prod
```

**The exam expects you to recognise** that `bundle validate` runs on every PR; `bundle deploy` runs on merge to main, scoped to each target. Both commands are the **same regardless of cloud or workspace** — the target manifest changes; the CLI does not.

## Run-as identity — service principals in non-dev

Bundles execute under an identity. Two questions to answer:

- **Who deploys the bundle?** The principal who runs `databricks bundle deploy`. In CI this is a service principal authenticated via OAuth M2M.
- **Who runs the deployed resources?** Configured by `run_as` on each target.

**Best practice the exam expects:**

- **dev** — `run_as: user` (the developer's identity).
- **test / prod** — `run_as: service_principal_name: <name>` so production jobs don't depend on any human's account.

**Permission boundary.** The service principal needs:

- **`USE CATALOG`** + **`USE SCHEMA`** on the catalogs / schemas it touches.
- **`SELECT` / `MODIFY`** on tables.
- **`CAN_MANAGE_RUN`** on the bundle's jobs / pipelines.
- **Workspace folder permissions** for the workspace path the bundle deploys into.

Notebook 09 walks through GRANT / REVOKE in detail.

## What belongs in a bundle, what doesn't

| Asset | In a bundle? |
|---|---|
| Lakeflow Jobs | ✅ |
| Lakeflow Spark Declarative Pipelines | ✅ |
| Notebooks invoked by the above | ✅ |
| Python source files / wheels | ✅ |
| ML experiments + models | ✅ |
| Volumes + schemas the bundle creates | ✅ |
| AI/BI dashboards | ✅ |
| Raw data files | ❌ — bundles aren't a data deployment tool |
| Unity Catalog catalogs themselves | ❌ — created out of band, then referenced |
| Service principals + IAM roles | ❌ — managed by your cloud / identity provider |
| Secrets values | ❌ — only secret *scope references* go in bundles |

**The rule:** if the asset is workspace-shaped code, put it in a bundle. If it's infrastructure (catalogs, principals, network) or data, keep it in Terraform / cloud IaC / data pipelines.

## Hands-on — a tiny bundle for the bank's pipeline

The cells below assemble a minimal bundle directory **on disk** that mirrors `fintech-platform`. The notebook itself is just emitting the file contents — you'd commit these to Git, install the CLI, and run `databricks bundle deploy -t dev` from your laptop or CI runner.

Run in a Databricks notebook or a local Jupyter — these cells only write files, no Databricks API calls.

In [ ]:
# 1) Lay out a minimal bundle tree under /tmp/fintech-platform.
from pathlib import Path

root = Path("/tmp/fintech-platform")
for sub in ("resources", "notebooks", "src", ".github/workflows"):
    (root / sub).mkdir(parents=True, exist_ok=True)

print("Tree:")
for p in sorted(root.rglob("*")):
    print("  ", p.relative_to(root))

In [ ]:
# 2) Write databricks.yml — the manifest. Variables, targets, mode, run_as.
MANIFEST = '''\
bundle:
  name: fintech-platform

variables:
  catalog:
    description: "Target UC catalog"
    default: fintech_dev
  node_type_id:
    description: "Worker node SKU"
    default: i3.xlarge

include:
  - resources/*.yml

targets:
  dev:
    mode: development
    workspace:
      host: https://dev.cloud.databricks.com
    variables:
      catalog: fintech_dev

  test:
    mode: development
    workspace:
      host: https://test.cloud.databricks.com
    variables:
      catalog: fintech_test

  prod:
    mode: production
    workspace:
      host: https://prod.cloud.databricks.com
    run_as:
      service_principal_name: fintech-platform-sp
    variables:
      catalog: fintech_prod
      node_type_id: i3.2xlarge
'''
(root / "databricks.yml").write_text(MANIFEST)
print((root / "databricks.yml").read_text())

In [ ]:
# 3) Write resources/job_nightly.yml — the Lakeflow Job, parametrised by ${var.catalog}.
JOB = '''\
resources:
  jobs:
    fintech_nightly_ingest:
      name: fintech_nightly_ingest
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "UTC"
      email_notifications:
        on_failure: ["data-platform-oncall@bank.example"]
      parameters:
        - name: catalog
          default: ${var.catalog}
      tasks:
        - task_key: ingest_cards
          notebook_task:
            notebook_path: ../notebooks/ingest_cards.ipynb
            base_parameters:
              catalog: ${var.catalog}
          job_cluster_key: shared_etl_cluster
          max_retries: 3
        - task_key: silver_build
          depends_on: [{task_key: ingest_cards}]
          notebook_task:
            notebook_path: ../notebooks/silver_build.ipynb
            base_parameters:
              catalog: ${var.catalog}
          job_cluster_key: shared_etl_cluster
        - task_key: notify
          depends_on: [{task_key: silver_build}]
          run_if: ALL_DONE
          notebook_task:
            notebook_path: ../notebooks/notify.ipynb
      job_clusters:
        - job_cluster_key: shared_etl_cluster
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            node_type_id: ${var.node_type_id}
            num_workers: 4
'''
(root / "resources/job_nightly.yml").write_text(JOB)
print((root / "resources/job_nightly.yml").read_text())

In [ ]:
# 4) Write resources/pipeline_card_etl.yml — the Lakeflow Spark Declarative Pipeline.
PIPELINE = '''\
resources:
  pipelines:
    card_etl:
      name: card_etl_${bundle.target}
      target: ${var.catalog}.gold
      libraries:
        - notebook:
            path: ../src/pipeline.py
      continuous: false
      development: true
      channel: CURRENT
'''
(root / "resources/pipeline_card_etl.yml").write_text(PIPELINE)
print((root / "resources/pipeline_card_etl.yml").read_text())

In [ ]:
# 5) Write the GitHub Actions deploy workflow.
GHA = '''\
name: Deploy bundle
on:
  pull_request: {}
  push:
    branches: [main]

jobs:
  validate:
    runs-on: ubuntu-latest
    env:
      DATABRICKS_HOST:          ${{ secrets.DATABRICKS_HOST_DEV }}
      DATABRICKS_CLIENT_ID:     ${{ secrets.DATABRICKS_CLIENT_ID }}
      DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle validate -t dev
      - run: databricks bundle validate -t test
      - run: databricks bundle validate -t prod

  deploy_test_then_prod:
    needs: validate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    env:
      DATABRICKS_CLIENT_ID:     ${{ secrets.DATABRICKS_CLIENT_ID }}
      DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - env: { DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_TEST }} }
        run: databricks bundle deploy -t test
      - env: { DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_TEST }} }
        run: databricks bundle run smoke_test_job -t test
      - env: { DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_PROD }} }
        run: databricks bundle deploy -t prod
'''
(root / ".github/workflows/deploy.yml").write_text(GHA)
print("Wrote .github/workflows/deploy.yml")

In [ ]:
# 6) The CLI cheat sheet you'd run from your laptop or CI.
print("""
# Install once
brew tap databricks/tap && brew install databricks   # macOS
# or curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

# Authenticate locally
databricks auth login --host https://dev.cloud.databricks.com

# Validate, deploy, run, destroy — substitute -t test / -t prod as appropriate
cd /tmp/fintech-platform
databricks bundle validate -t dev
databricks bundle deploy   -t dev
databricks bundle run      fintech_nightly_ingest -t dev
databricks bundle destroy  -t dev          # tear down an ephemeral env
""")

## Section 5 self-check

**1.** A team wants a modular way to deploy, version, and orchestrate ETL pipelines in Databricks — enabling CI/CD and repeatability. Which feature fits?

- A. Track pipelines as MLflow models in Unity Catalog
- B. Use Databricks Asset Bundles / Automation Bundles in Git, with `bundle deploy` for CI/CD
- C. Mount notebooks to DBFS and trigger via Jobs API v2
- D. Snapshot the workspace UI nightly

**2.** Which file is the entry point of an Automation Bundle?

- A. `pyproject.toml`
- B. `databricks.yml`
- C. `bundle.json`
- D. `dab.cfg`

**3.** A bundle target for production sets `mode: production` and `run_as: service_principal_name: fintech-platform-sp`. Why use a service principal in prod?

- A. Service principals run cheaper
- B. Production jobs should not depend on a human's account or permissions
- C. Service principals are required to run pipelines
- D. Personal access tokens are deprecated

**4.** Which CLI command parses the manifest, resolves variables for a target, checks references, and emits the JSON it would send to the API?

- A. `databricks bundle deploy`
- B. `databricks bundle validate`
- C. `databricks bundle init`
- D. `databricks bundle run`

**5.** Which is the **current** product name for *Databricks Repos*?

- A. Databricks Asset Bundles
- B. Databricks Git Folders
- C. Lakeflow Connect
- D. Workspace Files

<details><summary>Show answers</summary>

1. **B** — bundles are the modern packaging + CI/CD unit.
2. **B** — `databricks.yml` is the manifest at the bundle root.
3. **B** — production should run as a service principal, not a human.
4. **B** — `bundle validate` is the dry-run / lint step.
5. **B** — Repos was renamed Databricks Git Folders.

</details>

## Summary

- **Databricks Git Folders** (formerly Repos) = workspace folders backed by Git. Branch, commit, push, open PRs from the workspace UI.
- **Automation Bundles** (formerly Databricks Asset Bundles / DABs) package every resource as code, deployed via `databricks bundle deploy`.
- The **`databricks.yml`** manifest has three required sections: `bundle`, `resources`, `targets`. Plus optional `variables` and `include`.
- **Resources** are jobs, Lakeflow Spark Declarative Pipelines, models, dashboards, volumes, schemas — declared in YAML, referenced from the manifest.
- **Targets** (`dev`, `test`, `prod`) are named deployments with their own workspace, identity, and variable overrides.
- **Variables** + `${var.x}` + target-level overrides are how one codebase serves three environments.
- **`mode: production`** + **`run_as: service_principal_name: ...`** is the prod target pattern.
- **CLI**: `validate` (dry-run), `deploy` (push to workspace), `run` (kick off), `destroy` (tear down).
- **CI**: run `validate` on every PR; run `deploy` on merge to main, target by target.
- Bundles **don't** own catalogs, principals, raw data, or secret values — those are out-of-band infrastructure.

Next up: **notebook 08 — Troubleshooting, Monitoring & Optimization** — Spark UI, AQE, skew, spilling, Liquid Clustering, Predictive Optimization, cluster failures, and OOM diagnosis.